In [ ]:
from guardian_help import *

import numpy as np
import pandas as pd
import os
from scipy.io import loadmat

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

import pandas as pd
import numpy as np
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix, recall_score, precision_score, accuracy_score
from sklearn.ensemble import IsolationForest
import numpy as np
from sklearn.metrics import  f1_score

# Isolation forest
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import skew, kurtosis, entropy


path = "./data/"
# Path to the users folder
path_users = path + "users/"

window_path = {
    1: "features/window1000/",
    5: "features/window5000/",
    10: "features/window10000/",
    30: "features/window30000/",
}


BASELINE_TRAINING_RATIO = 0.7

DEBUG = False

# --- Hyperparameters ---
TEST_SIZE = 0.3
BASELINE_TRAINING_RATIO = 0.7
RANDOM_STATE = 42
N_ESTIMATORS = 100
CONTAMINATION_RATE = "auto"

feature_cols_kinematic = [
        "number_out_of_screen"
        "d_mean",
        "d_entropy",
        "v_mean",
        "v_entropy",
        "a_mean",
        "sampling_jitter",
    ]


feature_cols_kinematic = [
        "number_out_of_screen",
        "d_mean",
        "d_entropy",
        "v_mean",
        "v_entropy",
        "a_mean"
    ]


feature_cols_gaze = [
            #'number_fix', 
            'mean_duration_fix', 'entropy_fix', 
            'number_sac', 'duration_sac', 'velocity_sac', 'amplitude_sac']


users = [name for name in os.listdir(path + "users/") if os.path.isdir(os.path.join(path + "users/", name))]
print(users)

def get_gaze_features(window_path):

    df_gaze_set_aux = pd.DataFrame()

    # if path + window_path + "fixations/" does not exist, skip gaze features extraction
    if not os.path.exists(path + window_path + "fixations/"):
        print("Gaze features path does not exist, skipping gaze features extraction.")
        return df_gaze_set_aux

    for user in users:

        df_fixations = pd.read_csv(path + window_path + "fixations/" + user + ".csv")
        df_saccades = pd.read_csv(path + window_path + "saccades/"  + user + ".csv")

        # merge fixations and saccades dataframes on video column
        df_fix_sac = pd.merge(df_fixations, df_saccades, on=['video'], suffixes=('_fix', '_sac'))
        df_fix_sac['user'] = user
        # rename video column to video_number
        df_fix_sac = df_fix_sac.rename(columns={'video': 'video_number', 
                                            'mean duration': 'mean_duration_fix', 
                                            'entropy': 'entropy_fix',
                                            'duration': 'duration_sac',
                                            'amplitude': 'amplitude_sac',
                                            'velocity': 'velocity_sac',
                                            })
        #merge df_fix_sac with df_test on user and video_number
        df_gaze_set_aux = pd.concat([df_gaze_set_aux, df_fix_sac], ignore_index=True)

    return df_gaze_set_aux


 # Calculate the Euclidean distance between two points. 
def euclidean_distance_calculation(row):
    return math.sqrt((row['x'] - row['shifted_x']) ** 2 + (row['y'] - row['shifted_y']) ** 2)

def shannon_entropy(data):

    data = data.dropna()
    x_norm = (data - data.min()) / (data.max() - data.min())
    
    bins = np.unique(np.nanpercentile(data, np.linspace(0, 100, 21)))
    hist, _ = np.histogram(data, bins=bins)
    p = hist / hist.sum()
    p = p[p > 0]
    H = -np.sum(p * np.log2(p))

    #H_norm = H / np.log2(len(bins) - 1) # normalize entropy

    return H

def extract_kinematic_features(df_video):

    features = {}

    features["user"] = df_video.user.iloc[0]
    features["genre_category"] = df_video.genre_category.iloc[0]
    features["video_number"] = df_video.video_number.iloc[0]
    features["number_out_of_screen"] = df_video['number_out_of_screen'].sum()

    # distance between consecutive points
    features['d_mean'] = df_video['dist'].mean()
    features['d_max'] = df_video['dist'].max()
    features['d_std'] = df_video['dist'].std()

    # distance entropy - shannon entropy of the distance distribution
    features['d_entropy'] = shannon_entropy(df_video['dist'])

    dx = df_video['x'].diff().dropna()
    dy = df_video['y'].diff().dropna()
    angles = np.arctan2(dy, dx)
    # Use histogram to get probability distribution for entropy
    hist, _ = np.histogram(angles, bins=8, range=(-np.pi, np.pi), density=True)
    features['directional_entropy'] = entropy(hist + 1e-9) # Add epsilon to avoid log(0)
    
    # real velocity (already calculated as dist/dt)
    features['v_mean'] = df_video['real_velocity'].mean()
    features['v_max'] = df_video['real_velocity'].max()
    features['v_std'] = df_video['real_velocity'].std()
    features['v_skew'] = skew(df_video['real_velocity'])
    features['v_kurt'] = kurtosis(df_video['real_velocity'])

    # velocity entropy - shannon entropy of the velocity distribution
    features['v_entropy'] = shannon_entropy(df_video['real_velocity'])
    
    # Aceleración (Cambio de velocidad / dt)
    dv = df_video['real_velocity'].diff()
    acceleration = dv / (df_video['dt'] + 1e-9)
    features['a_mean'] = acceleration.mean()
    features['a_max'] = acceleration.max()
    features['a_std'] = acceleration.std()

    # acceleration entropy - shannon entropy of the acceleration distribution
    features['a_entropy'] = shannon_entropy(acceleration)

    features['sampling_jitter'] = df_video['dt'].std()
    features['hv_ratio'] = (df_video['x'].std() + 1e-9) / (df_video['y'].std() + 1e-9)

    # Calculate step-by-step displacements
    dx = df_video['x'].diff().dropna()
    dy = df_video['y'].diff().dropna()
    step_lengths = np.sqrt(dx**2 + dy**2)
    
    # 1. Distance Metrics
    total_path_length = step_lengths.sum()
    mean_step = step_lengths.mean()
    std_step = step_lengths.std()
    
    # 2. Tortuosity & Straightness
    # Direct distance between first and last point of the 1s window
    net_displacement = np.sqrt((df_video['x'].iloc[-1] - df_video['x'].iloc[0])**2 + 
                               (df_video['y'].iloc[-1] - df_video['y'].iloc[0])**2)
    
    # Avoid division by zero if the eye didn't move
    tortuosity = total_path_length / (net_displacement + 1e-9)
    straightness = (net_displacement + 1e-9) / (total_path_length + 1e-9)

    features['total_path_length'] = total_path_length
    features['mean_step'] = mean_step
    features['std_step'] = std_step
    features['path_tortuosity'] = tortuosity
    features['path_straightness'] = straightness
    
    return features

def get_kinematic_features(df_window):

    df_kinematic_features = pd.DataFrame()
    dict_kinematic_features = {}
    count = 0
    for user in users:

        # print percentage of users processed
        print(f"Processing user {user} ({users.index(user)+1}/{len(users)})", end='\r')

        df_user = df_window[df_window['user'] == user]


        for video_number in df_user['video_number'].unique():
            df_video = df_user[df_user['video_number'] == video_number]

            # SAMPLING_RATE = 30  # Hz
            # BIN_SIZE_MS = 1000 / SAMPLING_RATE  # ≈ 33.33 ms
            # df_video["bin"] = np.floor(df_video["timestamp"] / (BIN_SIZE_MS)) # Assign each row to a 30 Hz time bin
            # df_video = df_video.groupby("bin", as_index=False).first() # Keep the first row in each bin
            # df_video = df_video.drop(columns=["bin"])

            # print(f"User {user} - Original samples: {len(df_user[df_user['video_number'] == video_number])}, After resampling: {len(df_video)}")
            # print(df_video.timestamp.diff().describe())
            # Replace invalid values and interpolate
            df_video_nan = df_video.copy()
            df_video[['x', 'y']] = df_video[['x', 'y']].replace(-32768, np.nan)
            df_video[['x', 'y']] = df_video[['x', 'y']].replace(32768, np.nan)
            # df_video[['x', 'y']] = df_video[['x', 'y']].interpolate(method='linear', limit_direction='both')
            df_video["number_out_of_screen"] = df_video[['x', 'y']].isna().any(axis=1).astype(int)
            df_video.interpolate(inplace=True)
            

            # 1. Timestamp differences in seconds
            df_video['dt'] = df_video['timestamp'].diff() / 1000.0

            # 2. Euclidean Distance between consecutive points
            df_video['dx'] = df_video['x'].diff()
            df_video['dy'] = df_video['y'].diff()
            df_video['dist'] = np.sqrt(df_video['dx']**2 + df_video['dy']**2)

            # 3. Real Velocity = Distance / Time
            # We use a small epsilon to avoid division by zero if there are duplicate timestamps
            df_video['real_velocity'] = df_video['dist'] / (df_video['dt'] + 1e-9)

                
            # Extract robust features for this user and video
            features = extract_kinematic_features(df_video)
            
            # Append to the main DataFrame
            dict_kinematic_features[count] = features
            count += 1
    df_kinematic_features = pd.DataFrame.from_dict(dict_kinematic_features, orient='index')
    return df_kinematic_features

